# Поиск статей помощи по запросу пользователя

По тексту обращения нужно вернуть топ-10 статей базы помощи (793 статьи), метрика MAP@10.

Итог: 0.538 на калибровке, 0.531 на тесте.

Скор статьи считается как взвешенная сумма пяти сигналов, веса подобраны на калибровке:

```
score = E5 + 0.3*BM25 + 0.7*ColBERT + 0.15*CE + 0.8*BERTA
```

Ноутбук повторяет пайплайн из `solution/` и выдаёт тот же `answer.csv`, что загружен на платформу. Все тяжёлые шаги кэшируются на диск, поэтому повторный прогон быстрый. Считалось на MacBook M2 8 ГБ, пиковая память меньше 3 ГБ.

## Подготовка

Зависимости, веса моделей и предобработка корпуса (чистка HTML, лемматизация, кэш).

In [ ]:
import os, sys
os.chdir(os.path.dirname(os.path.abspath("__file__")))
sys.path.insert(0, "solution")

In [ ]:
!bash scripts/download_models.sh

In [ ]:
!PYTHONPATH=solution python solution/prep.py

Пример чистки HTML: вырезаются теги и сущности, сохраняются границы абзацев.

In [ ]:
from common import load_data, clean_html
art, cal, test = load_data()
print(art["body"].iloc[0][:200])
print("---")
print(clean_html(art["body"].iloc[0])[:200])

## Кодирование корпуса

Статьи режутся на чанки ~1100 символов с перекрытием 200, к чанку приклеивается заголовок. Дальше одноразовые батч-прогоны моделей с кэшем на диск:

- ColBERT-векторы чанков bge-m3 в двух вариантах длины: 256 токенов (для отбора кандидатов cross-encoder'у) и 384 (финальный сигнал);
- ColBERT-векторы запросов;
- BERTA-векторы чанков и запросов;
- cross-encoder скорит топ-20 кандидатов теста.

На холодном старте это самый долгий блок (порядка часа), с кэшами он проходит мгновенно.

In [ ]:
!PYTHONPATH=solution python solution/colbert.py encode_docs

In [ ]:
!CB_MAXLEN=384 PYTHONPATH=solution python solution/colbert.py encode_docs

In [ ]:
!PYTHONPATH=solution python solution/colbert.py encode_cal
!PYTHONPATH=solution python solution/colbert.py encode_test

In [ ]:
!PYTHONPATH=solution python solution/berta.py docs
!PYTHONPATH=solution python solution/berta.py cal
!PYTHONPATH=solution python solution/berta.py test

In [ ]:
!PYTHONPATH=solution python solution/ce_test.py

## Общие функции скоринга

Гибрид: e5-large по чанкам (среднее top-2 на статью) + BM25 по леммам. ColBERT: maxsim по токенам, максимум по чанкам статьи для топ-100 кандидатов гибрида. BERTA: те же чанки, среднее top-2.

In [ ]:
import os, hashlib, re
import numpy as np, pandas as pd
from rank_bm25 import BM25Okapi
from dense import encode, minmax, rank_from_scores
from common import mapk, parse_gt

CACHE = "solution/cache"
P = pd.read_pickle(CACHE + "/prep.pkl")
aid = np.array(P["article_id"]); nA = len(aid)
titles, bodies = P["art_title"], P["art_body_text"]
M = "intfloat/multilingual-e5-large"

CHARS, OV, MC = 1100, 200, 8
owner, chunks = [], []
for i in range(nA):
    b = bodies[i]
    if len(b) <= CHARS:
        pc = [b]
    else:
        pc, s = [], 0
        while s < len(b) and len(pc) < MC:
            pc.append(b[s:s+CHARS]); s += CHARS - OV
    for p in pc:
        chunks.append(f"{titles[i]}. {p}".strip()); owner.append(i)
owner = np.array(owner)
by_art = [[] for _ in range(nA)]
for c in range(len(owner)):
    by_art[owner[c]].append(c)

GRE = re.compile(r"(здравствуйте|добрый день|добрый вечер|доброе утро|привет|пожалуйста|подскажите|скажите|спасибо|доброго времени суток)[\s,!.]*", re.I)
key = hashlib.md5(("chunk|"+M+"|"+str(len(chunks))+chunks[0][:40]+chunks[-1][:40]).encode()).hexdigest()[:12]
cemb = np.load(f"{CACHE}/chunk_{key}.npy")

In [ ]:
def hybrid(split):
    clean = [GRE.sub("", q).strip() for q in P[f"{split}_text"]]
    qL = encode(M, clean, "query: ", batch=16)
    sims = qL @ cemb.T
    n_q = sims.shape[0]
    EL = np.zeros((n_q, nA))
    for a, cols in enumerate(by_art):
        cols = np.array(cols); sub = sims[:, cols]; tk = min(2, sub.shape[1])
        EL[:, a] = np.partition(sub, -tk, axis=1)[:, -tk:].mean(axis=1)
    bm = BM25Okapi(P["art_tokens"], k1=2.0, b=0.5)
    BM = np.array([bm.get_scores(q) for q in P[f"{split}_tokens"]])
    return minmax(EL) + 0.3 * minmax(BM), n_q

def colbert_scores(split, tag, hyb, n_q, k=100):
    fn = f"{CACHE}/{tag}_{split}_max_top{k}.npy"
    if os.path.exists(fn):
        return np.load(fn)
    dv = np.load(f"{CACHE}/{tag}_docs_vecs.npy", mmap_mode="r")
    doff = np.load(f"{CACHE}/{tag}_docs_offs.npy")
    qv = np.load(f"{CACHE}/cb_{split}q_vecs.npy")
    qoff = np.load(f"{CACHE}/cb_{split}q_offs.npy")
    cand = np.argsort(-hyb, axis=1)[:, :k]
    CB = np.zeros((n_q, nA))
    for qi in range(n_q):
        q = qv[qoff[qi]:qoff[qi+1]].astype(np.float32)
        for a in cand[qi]:
            best = -1e9
            for c in by_art[a]:
                d = dv[doff[c]:doff[c+1]].astype(np.float32)
                s = (q @ d.T).max(axis=1).mean()
                if s > best: best = s
            CB[qi, a] = best
    np.save(fn, CB)
    return CB

def berta_scores(split, n_q):
    de = np.load(f"{CACHE}/berta_docs.npy")
    qe = np.load(f"{CACHE}/berta_{split}q.npy")
    sims = qe @ de.T
    BE = np.zeros((n_q, nA))
    for a, cols in enumerate(by_art):
        cols = np.array(cols); sub = sims[:, cols]; tk = min(2, sub.shape[1])
        BE[:, a] = np.partition(sub, -tk, axis=1)[:, -tk:].mean(axis=1)
    return BE

Cross-encoder: mmarco-mMiniLMv2 скорит пары (запрос, лучший чанк статьи) для топ-20 кандидатов. Для теста скоры уже посчитаны скриптом `ce_test.py`, для калибровки при отсутствии кэша считаются здесь (порядка 40 минут на CPU).

In [ ]:
def ce_scores(split, hyb, n_q):
    fn = f"{CACHE}/ce_mini_{split}_top20.npy"
    if os.path.exists(fn):
        return np.load(fn)
    import torch
    torch.set_num_threads(4)
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    clean = [GRE.sub("", q).strip() for q in P[f"{split}_text"]]
    qL = encode(M, clean, "query: ", batch=16)
    sims = qL @ cemb.T
    best_chunk = np.zeros((n_q, nA), dtype=int)
    for a, cols in enumerate(by_art):
        cols = np.array(cols)
        best_chunk[:, a] = cols[np.argmax(sims[:, cols], axis=1)]
    CB256 = colbert_scores(split, "cb", hyb, n_q)
    base = hyb + 0.6 * minmax(CB256)
    cand = np.argsort(-base, axis=1)[:, :20]
    D = "solution/models/mmarco-minilm"
    tok = AutoTokenizer.from_pretrained(D)
    model = AutoModelForSequenceClassification.from_pretrained(D); model.eval()
    pairs, meta = [], []
    for qi in range(n_q):
        for a in cand[qi]:
            pairs.append((clean[qi], chunks[best_chunk[qi, a]])); meta.append((qi, a))
    sc = []
    with torch.no_grad():
        for i in range(0, len(pairs), 32):
            b = pairs[i:i+32]
            enc = tok([x[0] for x in b], [x[1] for x in b], padding=True, truncation=True, max_length=384, return_tensors="pt")
            sc.append(model(**enc).logits.reshape(-1).numpy())
    sc = np.concatenate(sc)
    RS = np.full((n_q, nA), -1e9)
    for (qi, a), s in zip(meta, sc):
        RS[qi, a] = s
    np.save(fn, RS)
    return RS

def final_score(split):
    hyb, n_q = hybrid(split)
    CB = colbert_scores(split, "cb384", hyb, n_q)
    RS = ce_scores(split, hyb, n_q)
    RSc = np.where(RS <= -1e8, np.nanmin(np.where(RS <= -1e8, np.nan, RS)), RS)
    BE = berta_scores(split, n_q)
    return hyb + 0.7*minmax(CB) + 0.15*minmax(RSc) + 0.8*minmax(BE), n_q

## Проверка качества на calibration.f

Метрика MAP@10, разметка есть только у калибровки. Здесь же вклад сигналов по шагам.

In [ ]:
gt = parse_gt(pd.Series(P["cal_gt"]))
hyb_cal, n_cal = hybrid("cal")
CB_cal = colbert_scores("cal", "cb384", hyb_cal, n_cal)
RS_cal = ce_scores("cal", hyb_cal, n_cal)
RSc_cal = np.where(RS_cal <= -1e8, np.nanmin(np.where(RS_cal <= -1e8, np.nan, RS_cal)), RS_cal)
BE_cal = berta_scores("cal", n_cal)

steps = {
    "e5-large + BM25": hyb_cal,
    "+ ColBERT": hyb_cal + 0.7*minmax(CB_cal),
    "+ CrossEncoder": hyb_cal + 0.7*minmax(CB_cal) + 0.15*minmax(RSc_cal),
    "+ BERTA (финал)": hyb_cal + 0.7*minmax(CB_cal) + 0.15*minmax(RSc_cal) + 0.8*minmax(BE_cal),
}
for name, S in steps.items():
    print(f"{name}: MAP@10 = {mapk(gt, rank_from_scores(S)):.4f}")

Краткий разбор ошибок: распределение AP по запросам и примеры полных промахов. Основная ошибка — путаница соседних статей одной темы.

In [ ]:
from common import apk
S_final = steps["+ BERTA (финал)"]
ranked = rank_from_scores(S_final)
aps = np.array([apk(set(g), r) for g, r in zip(gt, ranked)])
print(f"полных промахов (AP=0): {(aps==0).sum()} из {n_cal}")
print(f"идеальных (AP=1): {(aps==1).sum()} из {n_cal}")
art_idx = {a: i for i, a in enumerate(aid)}
for i in np.where(aps == 0)[0][:3]:
    print("---", P["cal_text"][i][:80].replace("\n", " "))
    print("   ожидалось:", [titles[art_idx[a]][:50] for a in gt[i] if a in art_idx])
    print("   получено: ", titles[art_idx[ranked[i][0]]][:50])

## Ответ для теста

Финальный скор на тестовых запросах и запись `answer.csv`.

In [ ]:
S_test, n_test = final_score("test")
test_id = P["test_id"]
rows = []
for qi in range(n_test):
    idx = np.argsort(-S_test[qi])[:10]
    rows.append((test_id[qi], " ".join(str(int(aid[i])) for i in idx)))
out = pd.DataFrame(rows, columns=["query_id", "answer"])
out.to_csv("solution/answer.csv", index=False)
print(len(out), "строк")
out.head()

In [ ]:
a = pd.read_csv("solution/answer.csv")
arts = set(aid.tolist())
ok = all(len(ids) == 10 and len(set(ids)) == 10 and all(i in arts for i in ids)
         for ids in (list(map(int, str(x).split())) for x in a.answer))
print("формат корректен:", ok)
print("все query_id теста покрыты:", set(a.query_id) == set(test_id))